# 01_Data_Collection
Sentinel→Landsat metadata pipeline.

In [ ]:
!pip -q install earthengine-api geemap pandas tqdm

import ee, pandas as pd
from tqdm import tqdm
from google.colab import files

ee.Authenticate()
ee.Initialize(project="heatwave-flood-prediction")

uploaded=files.upload()
filename=list(uploaded.keys())[0]
df=pd.read_csv(filename)

metadata=[]
skipped=[]

for _,row in tqdm(df.iterrows(),total=len(df)):
    event_id=row["event_id"]
    event_date=row["event_start_date"]
    lat=row["center_lat"]
    lon=row["center_lon"]

    point=ee.Geometry.Point([lon,lat])
    start=ee.Date(event_date).advance(-5,"day")
    end=ee.Date(event_date)

    try:
        s2=(ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
              .filterBounds(point)
              .filterDate(start,end)
              .sort("system:time_start",False))

        if s2.size().getInfo()>0:
            img=ee.Image(s2.first())
            metadata.append({
                "event_id":event_id,
                "event_date":event_date,
                "satellite":"Sentinel-2",
                "image_date":ee.Date(img.get("system:time_start")).format("YYYY-MM-dd").getInfo(),
                "latitude":lat,
                "longitude":lon
            })
            continue

        year=pd.to_datetime(event_date).year
        if year<=2021:
            ls=ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        else:
            ls=ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").merge(
                ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))

        ls=(ls.filterBounds(point)
              .filterDate(start,end)
              .sort("system:time_start",False))

        if ls.size().getInfo()>0:
            img=ee.Image(ls.first())
            metadata.append({
                "event_id":event_id,
                "event_date":event_date,
                "satellite":"Landsat",
                "image_date":ee.Date(img.get("system:time_start")).format("YYYY-MM-dd").getInfo(),
                "latitude":lat,
                "longitude":lon
            })
        else:
            skipped.append({"event_id":event_id,
                            "event_date":event_date,
                            "reason":"No image within previous 5 days"})
    except Exception as e:
        skipped.append({"event_id":event_id,
                        "event_date":event_date,
                        "reason":str(e)})

pd.DataFrame(metadata).to_csv("image_metadata.csv",index=False)
pd.DataFrame(skipped).to_csv("skipped_events.csv",index=False)

files.download("image_metadata.csv")
files.download("skipped_events.csv")
print("Done")
